In [11]:
from langchain_qdrant import QdrantVectorStore, FastEmbedSparse, RetrievalMode
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from pathlib import Path

# Ragas Evaluation
from ragas.testset import TestsetGenerator
from ragas import evaluate, EvaluationDataset
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, AnswerRelevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper 

from tqdm import tqdm
import os

from dotenv import load_dotenv
load_dotenv()

/var/folders/73/489qwk890nz19z7w2q63j4k00000gn/T/ipykernel_45707/1390594150.py:11: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, AnswerRelevancy
/var/folders/73/489qwk890nz19z7w2q63j4k00000gn/T/ipykernel_45707/1390594150.py:11: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, AnswerRelevancy
/var/folders/73/489qwk890nz19z7w2q63j4k00000gn/T/ipykernel_45707/1390594150.py:11: DeprecationWarning: Importing FactualCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please u

True

In [3]:
path = './datasets/documents/MT RESEARCH FINAL.pdf'
docs = PyPDFLoader(path).load()

print(f"Total number of pages: {len(docs)}")

Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)
Ignoring wrong pointing object 14 0 (offset 0)
Ignoring wrong pointing object 20 0 (offset 0)
Ignoring wrong pointing object 22 0 (offset 0)
Ignoring wrong pointing object 24 0 (offset 0)
Ignoring wrong pointing object 26 0 (offset 0)
Ignoring wrong pointing object 28 0 (offset 0)
Ignoring wrong pointing object 30 0 (offset 0)
Ignoring wrong pointing object 38 0 (offset 0)
Ignoring wrong pointing object 44 0 (offset 0)
Ignoring wrong pointing object 51 0 (offset 0)
Ignoring wrong pointing object 78 0 (offset 0)


Total number of pages: 21


In [13]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,  
    add_start_index=True,

)
split_docs = text_splitter.split_documents(docs)
split_docs[0].page_content


'Artificial Intelligence & Data-Driven Technologies in Modern Tourism: A Comparative Study (India and Global Perspectives) 1. Abstract This study examines the impact of Artificial Intelligence (AI) and data-driven technologies on the modern tourism industry, with a comparative focus on India and leading global economies such as the United States, China, Japan, Singapore, and the UAE. The research analyzes recent academic literature, government reports, and industry case studies (2020–2026) to understand how AI is transforming tourism operations, economic development, and customer experiences worldwide.  The key objective of this paper is to evaluate emerging AI applications in tourism, identify gaps between technological potential and practical implementation, and compare India’s progress with global leaders. Major AI use cases include personalized recommendation, AI chatbots for booking and customer support, predictive pricing models, smart infrastructure, virtual and augmented realit

In [18]:
embedding_model = OpenAIEmbeddings(model="text-embedding-3-large")

QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")

vector_store = QdrantVectorStore.from_documents(
    documents=split_docs,
    url="https://9219ef2c-5862-409f-93ef-f0808298aad3.eu-west-2-0.aws.cloud.qdrant.io",
    api_key=QDRANT_API_KEY,
    collection_name="traditional_rag_docs",
    embedding=embedding_model,
)
print("Documents have been embedded and stored in Qdrant.")

Documents have been embedded and stored in Qdrant.


In [21]:
query = "What are the main research objectives and what are actually covered in the paper?"

retriever = vector_store.similarity_search_with_relevance_scores(
    query=query,
    k=5
)
for doc,score in retriever:
    print(f"Score: {score:.4f}")
    print(f"Content: {doc.page_content}")
    print("\n-------\n")

Score: 0.7575
Content: 3. Research Objectives This research is guided by the following objectives: • Objective 1: Identify the latest AI and data-driven technological trends in the tourism industry globally (last 5–10 years), including tools like machine learning, chatbots, IoT, VR/AR, and big data analytics. • Objective 2: Analyze key innovations and their impacts in tourism, linking them to software engineering concepts (e.g. ML models, cloud systems, API integrations). • Objective 3: Compare the implementation and outcomes of these technologies in India versus international contexts, highlighting examples and case studies from each. • Objective 4: Assess user behavior and economic trends (e.g. market growth, traveler attitudes toward AI) to understand how demand is shaping innovation. • Objective 5: Identify gaps in current research and practice, suggesting areas needing further study (e.g. ethics, accessibility, integration). These objectives ensure a structured examination: we fir

In [22]:
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

context_text = "\n\n---\n\n".join([doc.page_content for doc, _ in retriever])

prompt = f""" 
You are an expert AI assistant for deep learning and machine learning.
Answer the question using ONLY the provided context below.

Instructions:
- Base your answer strictly on the context; do not add outside knowledge
- Be precise and thorough — cover all relevant aspects mentioned in the context
- Use clear structure (bullet points or paragraphs) when the answer has multiple parts
- If the context does not contain enough information to answer fully, say so explicitly

Context:
{context_text}

Question: {query}

Answer
"""

response = llm.invoke(prompt).content
print(response)

The main research objectives and what is actually covered in the paper are as follows:

**Main Research Objectives:**

1. **Identify Latest AI and Data-Driven Trends in Tourism Globally (Last 5–10 Years):**  
   - Focus on technologies such as machine learning, chatbots, IoT, VR/AR, and big data analytics.

2. **Analyze Key Innovations and Their Impacts in Tourism:**  
   - Link innovations to software engineering concepts like ML models, cloud systems, and API integrations.

3. **Compare Implementation and Outcomes in India vs. International Contexts:**  
   - Highlight examples and case studies from India and other countries to reveal contextual differences.

4. **Assess User Behavior and Economic Trends:**  
   - Study market growth and traveler attitudes toward AI to understand how demand shapes innovation.

5. **Identify Gaps in Current Research and Practice:**  
   - Suggest areas needing further study, including ethics, accessibility, and technology integration.

---

**What Is 

In [23]:
def traditional_rag_query(query: str, k: int = 5, verbose: bool = True):
    results = vector_store.similarity_search_with_relevance_scores(query=query, k=k)

    contexts = [doc.page_content for doc, _ in results]
    context_text = "\n\n---\n\n".join(contexts)

    prompt = f"""You are an expert AI assistant for deep learning and machine learning.
Answer the question using ONLY the provided context below.

Instructions:
- Base your answer strictly on the context; do not add outside knowledge
- Be precise and thorough — cover all relevant aspects mentioned in the context
- Use clear structure (bullet points or paragraphs) when the answer has multiple parts
- If the context does not contain enough information to answer fully, say so explicitly

Context:
{context_text}

Question: {query}

Answer:"""

    response = llm.invoke(prompt).content

    if verbose:
        print(f"\n{'='*50}\nANSWER:\n{response}\n{'='*50}")

    return {
        "answer": response,
        "sources": contexts,
        "query_used": query,
    }

print("traditional_rag_query ready")

traditional_rag_query ready


### Generate Test Dataset (Ragas)

In [24]:
import pandas as pd

test_data = [
    {
        "user_input": "What was the projected CAGR of the global AI in travel market from 2024 to 2029?",
        "reference": "The global AI in travel market was projected to grow at a CAGR of approximately 33.8%, increasing from $123.72 billion in 2024 to an estimated $531.95 billion by 2029."
    },
    {
        "user_input": "How much has India pledged for AI investment?",
        "reference": "India pledged $1.25 billion for AI investment."
    },
    {
        "user_input": "How does Heathrow Airport's AI adoption compare to Delhi Airport, and what does this reveal about the broader gap between global and Indian aviation?",
        "reference": "Heathrow's AI chatbot Hallie resolves around 90% of queries without human intervention, while Delhi Airport still relies more on human staff despite having kiosks and mobile apps. This reflects the broader pattern where resource-rich international airports have fully automated customer service, while Indian airports are still scaling basic digital infrastructure."
    },
    {
        "user_input": "How have AI-driven personalization strategies evolved differently in India versus global platforms like Booking.com?",
        "reference": "Globally, Booking.com has long used mature AI algorithms with a large user base and released a Global AI Sentiment Report showing high traveler demand. In India, MakeMyTrip integrated OpenAI's GPT API more recently to enable chat-to-book experiences, rapidly adapting to multilingual Indian markets. While both use similar underlying AI, Booking.com benefits from earlier maturity whereas MakeMyTrip is catching up with localization as its edge."
    },
    {
        "user_input": "What AI robot did Japan showcase at Expo 2025 Osaka?",
        "reference": "Japan showcased Miraia Lynx, an AI concierge robot at Expo 2025 Osaka that provides multilingual assistance and health monitoring."
    },
    {
        "user_input": "What percentage of travelers were excited about AI according to Booking.com's survey?",
        "reference": "91% of travelers were excited about AI and 89% wanted to use AI for travel planning, however only 6% fully trusted AI and 91% had privacy concerns."
    },
    {
        "user_input": "Considering India's digital infrastructure strengths like Aadhaar and UPI alongside its AI investment limitations, what is its realistic potential in AI-powered tourism?",
        "reference": "India's potential lies in leapfrogging traditional stages by coupling AI with its unique public-digital infrastructure like Aadhaar and UPI, which no other country matches in scale. However, it is held back by modest AI investment of $1.25 billion compared to China's multibillion initiatives, uneven broadband, regulatory uncertainties, and a large unorganized accommodation sector. The coming years will determine how effectively India translates its policy vision into on-the-ground solutions."
    },
    {
        "user_input": "How does the role of government policy differ in driving AI tourism adoption between Singapore, UAE, and India?",
        "reference": "Singapore uses its National AI Strategy 2.0 and Smart Nation policies to actively partner with tech firms like OpenAI through the STB MOU, enabling rapid system-wide implementation due to its compact geography. UAE drives adoption through the AI 2031 strategy backed by sovereign investment, achieving innovations like citywide contactless hotel check-in. India relies on broader national programs like Digital India and IIDP but faces slower ground-level implementation due to infrastructure gaps and regulatory uncertainties, making its approach more aspirational than operational compared to Singapore and UAE."
    },
    {
        "user_input": "What does a 5% improvement in demand prediction mean financially for hotels?",
        "reference": "A 5% improvement in demand prediction can increase hotel revenue by 2 to 3%."
    },
    {
        "user_input": "What is the Digistan plan in India?",
        "reference": "The Digistan plan is India's digital heritage initiative aimed at creating 3D models of monuments, though as of 2025 it remains only partially implemented."
    },
    {
        "user_input": "How does China leverage scale in its AI tourism strategy compared to smaller economies like Singapore?",
        "reference": "China leverages data from hundreds of millions of trips to train AI models at a scale no smaller economy can match, leading in AI publications and patents and applying AI across landmarks with facial recognition and smart city integration. Singapore compensates for its smaller scale through strategic national coordination, high digital penetration, and compact geography that allows rapid system-wide implementation through partnerships like the STB-OpenAI MOU. Both succeed but through fundamentally different approaches — China through volume, Singapore through precision."
    },
    {
        "user_input": "What are the key research gaps identified in the paper?",
        "reference": "The paper identifies five gaps: lack of systematic comparative analysis of AI tourism adoption across countries, limited empirical data quantifying actual ROI and benefits, absence of research on integrated multi-AI platforms, insufficient ethical and policy guidance especially for culturally diverse regions, and sparse understanding of specific user segments like rural versus urban Indian tourists."
    },
]

testset_df = pd.DataFrame(test_data)
print(f"Testset: {len(testset_df)} samples")
testset_df

Testset: 12 samples


,user_input,reference
0,What was the projected CAGR of the global AI i...,The global AI in travel market was projected t...
1,How much has India pledged for AI investment?,India pledged $1.25 billion for AI investment.
2,How does Heathrow Airport's AI adoption compar...,Heathrow's AI chatbot Hallie resolves around 9...
3,How have AI-driven personalization strategies ...,"Globally, Booking.com has long used mature AI ..."
4,What AI robot did Japan showcase at Expo 2025 ...,"Japan showcased Miraia Lynx, an AI concierge r..."
5,What percentage of travelers were excited abou...,91% of travelers were excited about AI and 89%...
6,Considering India's digital infrastructure str...,India's potential lies in leapfrogging traditi...
7,How does the role of government policy differ ...,Singapore uses its National AI Strategy 2.0 an...
8,What does a 5% improvement in demand predictio...,A 5% improvement in demand prediction can incr...
9,What is the Digistan plan in India?,The Digistan plan is India's digital heritage ...


### Run Pipeline on All Test Questions

In [25]:
test_questions = testset_df["user_input"].tolist()
ground_truths = testset_df["reference"].tolist()

responses = []
retrieved_contexts = []

for question in tqdm(test_questions, desc="Running Traditional RAG"):
    result = traditional_rag_query(query=question, k=5, verbose=False)
    responses.append(result["answer"])
    retrieved_contexts.append(result["sources"])

print(f"\nCompleted {len(responses)} queries")

Running Traditional RAG: 100%|██████████| 12/12 [01:33<00:00,  7.81s/it]


Completed 12 queries


### Ragas Evaluation

In [27]:
eval_list = [
    {"user_input": q, "response": r, "retrieved_contexts": c, "reference": g}
    for q, r, c, g in zip(test_questions, responses, retrieved_contexts, ground_truths)
]

evaluation_dataset = EvaluationDataset.from_list(eval_list)
evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini",n=3))

print("Running Ragas evaluation...")
result = evaluate(
    dataset=evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness(), AnswerRelevancy()],
    llm=evaluator_llm,
)

print("\n" + "-" * 50)
print("Ragas Evaluation Results (Traditional RAG):")
print(result)
print("=" * 50)

/var/folders/73/489qwk890nz19z7w2q63j4k00000gn/T/ipykernel_45707/1173469807.py:7: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini",n=3))


Running Ragas evaluation...


Evaluating:   4%|▍         | 2/48 [00:06<02:25,  3.16s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating:  21%|██        | 10/48 [00:44<02:46,  4.38s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating:  50%|█████     | 24/48 [01:22<00:57,  2.41s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating:  60%|██████    | 29/48 [01:35<00:46,  2.47s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating: 100%|██████████| 48/48 [03:28<00:00,  4.34s/it]



--------------------------------------------------
Ragas Evaluation Results (Traditional RAG):
{'context_recall': 0.8889, 'faithfulness': 0.9833, 'factual_correctness(mode=f1)': 0.6158, 'answer_relevancy': 0.8904}


In [ ]:
result_df = result.to_pandas()
print(result_df[["user_input", "context_recall", "faithfulness", "factual_correctness(mode=f1)", "answer_relevancy"]].to_string())